# 01 — Perfilado inicial (Entrega 2, dataset v2)

Objetivo: verificar volúmenes, tipos, nulos, cardinalidad e integridad referencial
del nuevo dataset multi-hogar (10 hogares, 90 días, ~25k eventos).
Las cifras de este notebook alimentan `docs/data_dictionary.md`.

Esquema actualizado:
- `movements_raw.csv`: 10 columnas (event_type, classification en lugar de action_type/location)
- `catalog_raw.csv`: category = Instacart dept_id numérico (6 valores)
- `inventory_v1.csv`: 17 columnas, generado por `src/preprocessing.py`


In [ ]:
import pandas as pd

catalog = pd.read_csv('../data/raw/catalog_raw.csv')
movements = pd.read_csv('../data/raw/movements_raw.csv', parse_dates=['timestamp'])

print(f'Catálogo:    {catalog.shape[0]} filas × {catalog.shape[1]} columnas')
print(f'Movimientos: {movements.shape[0]} filas × {movements.shape[1]} columnas')

## Catálogo — tipos, nulos y cardinalidad

In [ ]:
pd.DataFrame({
    'dtype': catalog.dtypes.astype(str),
    'nulos': catalog.isnull().sum(),
    'cardinalidad': catalog.nunique(),
})

In [ ]:
# Distribución de nutriscore y categorías
print('Nutriscore:', catalog['nutriscore'].value_counts(dropna=False).to_dict())
print(f"Categorías distintas (dept_id): {catalog['category'].nunique()}")
print('Dept_ids:', catalog['category'].value_counts().to_dict())


In [ ]:
# Detección de outlier en calories_100g (umbral 900 kcal)
outliers = catalog[catalog['calories_100g'] > 900]
print(f'Outliers calories_100g > 900: {len(outliers)}')
# Si 0 outliers: salvaguarda activa, nueva fuente ya no los contiene
outliers[['product_id', 'product_name', 'calories_100g']] if len(outliers) > 0 else 'Sin outliers en nueva fuente'


## Movimientos — tipos, nulos y cardinalidad

In [ ]:
pd.DataFrame({
    'dtype': movements.dtypes.astype(str),
    'nulos': movements.isnull().sum(),
    'cardinalidad': movements.nunique(),
})

In [ ]:
print('Rango temporal:', movements['timestamp'].min(), '→', movements['timestamp'].max())
print('Hogares únicos:', movements['household_id'].nunique())
print('event_type:', movements['event_type'].value_counts().to_dict())
print('classification:', movements['classification'].value_counts().to_dict())


## Integridad referencial

In [ ]:
huerfanos = (~movements['product_id'].astype(str).isin(catalog['product_id'].astype(str))).sum()
print(f'Eventos con product_id ausente en el catálogo: {huerfanos}')

## Nulos y completitud en `expiry_date`

En el nuevo esquema, `expiry_date` está presente en todos los eventos (0 nulos).
La columna `classification` distingue el motivo del evento de salida (Consumption/Waste/Forced_Waste).


In [ ]:
print('Nulos en expiry_date:', movements['expiry_date'].isnull().sum())
print('\nDistribución classification (motivo de salida):')
movements.groupby('event_type')['classification'].value_counts()
